In [1]:
import numpy as np
import pandas as pd

# Stal se mi nějaký problém při nahrávání z Pycharmu do GitHubu vytvořený nějakým nešikovným mergerováním verzí. Zkusím to odevzdat takhle, snad to půjde...

In [2]:
# 1) Načtení a příprava dat

# Task 1.1: Načtení CSV souborů do DataFramů 'df' a 'customer_info'
df = pd.read_csv("data/purchases.csv")
customer_info = pd.read_csv("data/customer_info.csv")

# Zamíchání řádků v 'df' ihned po načtení
df = df.sample(frac=1, random_state=42)

# Task 1.2: Převod sloupce 'Timestamp' v 'df' na formát datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Task 1.3: Nastavení 'Timestamp' jako indexu 'df' a vzestupné seřazení
df = df.set_index("Timestamp").sort_index()

print(df.head())
print(df.info())
print(customer_info.head())

                    Customer_id   Price
Timestamp                              
2025-01-01 00:00:00        C037  436.71
2025-01-01 00:30:00        C049  274.70
2025-01-01 01:00:00        C017  380.42
2025-01-01 01:30:00        C049  378.28
2025-01-01 02:00:00        C002   13.88
<class 'pandas.DataFrame'>
DatetimeIndex: 3000 entries, 2025-01-01 00:00:00 to 2025-03-04 11:30:00
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer_id  3000 non-null   str    
 1   Price        3000 non-null   float64
dtypes: float64(1), str(1)
memory usage: 70.3 KB
None
  Customer_id   Born_date       City
0        C001  1998-12-03  Frankfurt
1        C002  1988-07-03  Frankfurt
2        C003  1974-12-01     Munich
3        C004  2002-08-27  Frankfurt
4        C005  1967-03-05    Cologne


In [3]:
# 2) Spojení dat

# Task 2.1 & 2.2: Spojení nákupů s metadaty zákazníků podle 'Customer_id'
# Pomocí reset_index() a set_index("Timestamp") zachováme časový index i jeho seřazení
df_merged = (
    df.reset_index()
    .merge(customer_info[["Customer_id", "Born_date", "City"]], on="Customer_id", how="left")
    .set_index("Timestamp")
)

print(df_merged.head())
print(df_merged.info())

                    Customer_id   Price   Born_date       City
Timestamp                                                     
2025-01-01 00:00:00        C037  436.71  1996-09-14     Munich
2025-01-01 00:30:00        C049  274.70  1979-01-20     Munich
2025-01-01 01:00:00        C017  380.42  1981-02-23    Cologne
2025-01-01 01:30:00        C049  378.28  1979-01-20     Munich
2025-01-01 02:00:00        C002   13.88  1988-07-03  Frankfurt
<class 'pandas.DataFrame'>
DatetimeIndex: 3000 entries, 2025-01-01 00:00:00 to 2025-03-04 11:30:00
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer_id  3000 non-null   str    
 1   Price        3000 non-null   float64
 2   Born_date    3000 non-null   str    
 3   City         3000 non-null   str    
dtypes: float64(1), str(3)
memory usage: 117.2 KB
None


In [4]:
# 3) Groupby a agregace

# Task 3.1: Agregace podle Customer_id
customer_summary = df_merged.groupby("Customer_id").agg(
    total_revenue=("Price", "sum"),
    purchase_count=("Price", "count"),
    avg_price=("Price", "mean")
)

print("Task 3.1: Souhrn podle zákazníků (ukázka)")
print(customer_summary.head())

# Task 3.2: Extrakce názvu měsíce z indexu Timestamp
df_merged["Month"] = df_merged.index.month_name()

print("\nTask 3.2: Počet nákupů v jednotlivých měsících")
print(df_merged["Month"].value_counts())

# Task 3.3: Celkové měsíční tržby podle města (City a Month)
city_monthly_revenue = df_merged.groupby(["City", "Month"]).agg(
    total_revenue=("Price", "sum")
)

print("\nTask 3.3: Měsíční tržby podle měst")
print(city_monthly_revenue)

Task 3.1: Souhrn podle zákazníků (ukázka)
             total_revenue  purchase_count   avg_price
Customer_id                                           
C001              13055.75              56  233.138393
C002              15927.80              64  248.871875
C003              14947.34              56  266.916786
C004              15111.43              59  256.125932
C005              16798.61              64  262.478281

Task 3.2: Počet nákupů v jednotlivých měsících
Month
January     1488
February    1344
March        168
Name: count, dtype: int64

Task 3.3: Měsíční tržby podle měst
                    total_revenue
City      Month                  
Berlin    February       86163.85
          January       101192.17
          March          10474.86
Cologne   February       86781.31
          January        98030.00
          March          11220.64
Frankfurt February       52020.01
          January        61492.29
          March           6698.88
Hamburg   February       42056.1

In [5]:
# 4) Rolling Window Analysis

# Seřazení podle Customer_id a časového indexu Timestamp (chronologicky po jednotlivých zákaznících)
df_rolling = df_merged.sort_values(["Customer_id", "Timestamp"]).copy()

# Klouzavý průměr z 3 nákupů v rámci skupiny Customer_id
df_rolling["rolling_avg_3"] = (
    df_rolling.groupby("Customer_id")["Price"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

# Rozdíl ceny oproti předchozímu nákupu daného zákazníka
df_rolling["price_diff"] = df_rolling.groupby("Customer_id")["Price"].diff()

# Kontrola výsledku na prvních 6 nákupech zákazníka C001
print(
    df_rolling[df_rolling["Customer_id"] == "C001"][
        ["Customer_id", "Price", "rolling_avg_3", "price_diff"]
    ].head(6)
)

                    Customer_id   Price  rolling_avg_3  price_diff
Timestamp                                                         
2025-01-01 05:30:00        C001  376.61            NaN         NaN
2025-01-02 02:00:00        C001   63.23            NaN     -313.38
2025-01-02 10:00:00        C001  108.83         182.89       45.60
2025-01-05 05:00:00        C001   70.19          80.75      -38.64
2025-01-06 07:00:00        C001  132.80         103.94       62.61
2025-01-06 07:30:00        C001  338.93         180.64      206.13


In [6]:
# 5) Filtrování

# Nejprve vyfiltrujeme metadata pouze pro zákazníky z Berlína
berlin_customers = customer_info.loc[
    customer_info["City"] == "Berlin", ["Customer_id", "Born_date", "City"]
]

# Dále spojíme nákupy s berlínskými zákazníky pomocí inner join
df_berlin = (
    df.reset_index()
    .merge(berlin_customers, on="Customer_id", how="inner")
    .set_index("Timestamp")
    .sort_values(["Customer_id", "Timestamp"])
)

# Výpočty jako v části 4)
df_berlin["rolling_avg_3"] = (
    df_berlin.groupby("Customer_id")["Price"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)
df_berlin["price_diff"] = df_berlin.groupby("Customer_id")["Price"].diff()

print("Počet řádků:", len(df_berlin), "| Počet zákazníků:", df_berlin["Customer_id"].nunique())
print(df_berlin.head(6))

Počet řádků: 777 | Počet zákazníků: 13
                    Customer_id   Price   Born_date    City  rolling_avg_3  \
Timestamp                                                                    
2025-01-05 02:30:00        C013  474.31  1995-06-12  Berlin            NaN   
2025-01-06 00:00:00        C013  410.01  1995-06-12  Berlin            NaN   
2025-01-07 23:00:00        C013  396.90  1995-06-12  Berlin     427.073333   
2025-01-08 22:00:00        C013   92.70  1995-06-12  Berlin     299.870000   
2025-01-09 04:00:00        C013  392.28  1995-06-12  Berlin     293.960000   
2025-01-10 03:00:00        C013   34.44  1995-06-12  Berlin     173.140000   

                     price_diff  
Timestamp                        
2025-01-05 02:30:00         NaN  
2025-01-06 00:00:00      -64.30  
2025-01-07 23:00:00      -13.11  
2025-01-08 22:00:00     -304.20  
2025-01-09 04:00:00      299.58  
2025-01-10 03:00:00     -357.84  


In [7]:
# Další část k 5) aktuální nákup o více než 20 % dražší než předchozí nákup zákazníka
# Seřadíme původní nákupy chronologicky po zákaznících a nejprve vyfiltrujeme
df_sorted = df.sort_values(["Customer_id", "Timestamp"]).copy()
prev_price = df_sorted.groupby("Customer_id")["Price"].shift(1)
df_jump_only = df_sorted[df_sorted["Price"] > 1.20 * prev_price]

# K vyfiltrovaným řádkům připojíme metadata zákazníků
df_price_jump = (
    df_jump_only.reset_index()
    .merge(customer_info[["Customer_id", "Born_date", "City"]], on="Customer_id", how="left")
    .set_index("Timestamp")
    .sort_values(["Customer_id", "Timestamp"])
)

# Výpočty jako v části 4)
df_price_jump["rolling_avg_3"] = (
    df_price_jump.groupby("Customer_id")["Price"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)
df_price_jump["price_diff"] = df_price_jump.groupby("Customer_id")["Price"].diff()

print("\nPočet nákupů s nárůstem > 20 %:", len(df_price_jump))
print(df_price_jump.head(6))


Počet nákupů s nárůstem > 20 %: 1208
                    Customer_id   Price   Born_date       City  rolling_avg_3  \
Timestamp                                                                       
2025-01-02 10:00:00        C001  108.83  1998-12-03  Frankfurt            NaN   
2025-01-06 07:00:00        C001  132.80  1998-12-03  Frankfurt            NaN   
2025-01-06 07:30:00        C001  338.93  1998-12-03  Frankfurt     193.520000   
2025-01-06 11:30:00        C001  415.65  1998-12-03  Frankfurt     295.793333   
2025-01-07 19:30:00        C001  382.14  1998-12-03  Frankfurt     378.906667   
2025-01-11 15:30:00        C001  210.98  1998-12-03  Frankfurt     336.256667   

                     price_diff  
Timestamp                        
2025-01-02 10:00:00         NaN  
2025-01-06 07:00:00       23.97  
2025-01-06 07:30:00      206.13  
2025-01-06 11:30:00       76.72  
2025-01-07 19:30:00      -33.51  
2025-01-11 15:30:00     -171.16  


In [8]:
# 6) Lambda funkce

# Filtrování: zákazníci, kteří v únoru utratili více než 7 000
feb_high_spenders = df_merged.groupby("Customer_id").filter(
    lambda g: g[g.index.month == 2]["Price"].sum() > 7000
)

print("Počet zákazníků s útratou > 7 000 v únoru:", feb_high_spenders["Customer_id"].nunique())
print("Počet jejich nákupů celkem:", len(feb_high_spenders))

# Transformace: rozdíl ceny každého nákupu od průměrné útraty daného zákazníka
df_lambda = df_merged.copy()
df_lambda["diff_from_avg"] = df_lambda.groupby("Customer_id")["Price"].transform(
    lambda x: x - x.mean()
)

print(df_lambda[["Customer_id", "Price", "diff_from_avg"]].head())

Počet zákazníků s útratou > 7 000 v únoru: 19
Počet jejich nákupů celkem: 1170
                    Customer_id   Price  diff_from_avg
Timestamp                                             
2025-01-01 00:00:00        C037  436.71     173.064930
2025-01-01 00:30:00        C049  274.70      29.825135
2025-01-01 01:00:00        C017  380.42     120.337031
2025-01-01 01:30:00        C049  378.28     133.405135
2025-01-01 02:00:00        C002   13.88    -234.991875


In [9]:
# 7) Time-Based Analysis

# Měsíční míra růstu tržeb na zákazníka
monthly_revenue = (
    df_merged.groupby(["Customer_id", df_merged.index.to_period("M")])["Price"]
    .sum()
    .reset_index(name="revenue")
    .rename(columns={"Timestamp": "Month"})
    .sort_values(["Customer_id", "Month"])
)

prev_revenue = monthly_revenue.groupby("Customer_id")["revenue"].shift(1)
monthly_revenue["growth_rate"] = (monthly_revenue["revenue"] - prev_revenue) / prev_revenue

print("Měsíční tržby a míra růstu (prvních 9 řádků)")
print(monthly_revenue.head(9))

# Kumulativní součet nákupů v čase pro každého zákazníka
df_time = df_merged.sort_values(["Customer_id", "Timestamp"]).copy()
df_time["cumulative_revenue"] = df_time.groupby("Customer_id")["Price"].cumsum()

print("\nKumulativní součet nákupů pro zákazníka C001 (prvních 5 řádků)")
print(df_time[df_time["Customer_id"] == "C001"][["Customer_id", "Price", "cumulative_revenue"]].head())

Měsíční tržby a míra růstu (prvních 9 řádků)
  Customer_id    Month  revenue  growth_rate
0        C001  2025-01  7302.10          NaN
1        C001  2025-02  5204.79    -0.287220
2        C001  2025-03   548.86    -0.894547
3        C002  2025-01  7538.37          NaN
4        C002  2025-02  7838.50     0.039814
5        C002  2025-03   550.93    -0.929715
6        C003  2025-01  7133.09          NaN
7        C003  2025-02  7141.33     0.001155
8        C003  2025-03   672.92    -0.905771

Kumulativní součet nákupů pro zákazníka C001 (prvních 5 řádků)
                    Customer_id   Price  cumulative_revenue
Timestamp                                                  
2025-01-01 05:30:00        C001  376.61              376.61
2025-01-02 02:00:00        C001   63.23              439.84
2025-01-02 10:00:00        C001  108.83              548.67
2025-01-05 05:00:00        C001   70.19              618.86
2025-01-06 07:00:00        C001  132.80              751.66
